In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import joblib, os

df = pd.read_csv('../data/clean_matches.csv')
print("Shape:", df.shape)
print("Colonnes:", df.columns.tolist())
df.head()

Shape: (1260, 16)
Colonnes: ['id', 'name', 'category', 'round', 'round_name', 'index', 'played_at', 'status', 'score', 'winner', 'duration_minutes', 'players', 'competition_source', 'season_year', 'tournament_name', 'court']


,id,name,category,round,round_name,index,played_at,status,score,winner,duration_minutes,players,competition_source,season_year,tournament_name,court
0,6987,Tapia/Coello - Chingotto/Galan,men,1,Finals,0,2025-12-14,finished,"[{'team_1': '6(4)', 'team_2': '7'}, {'team_1':...",team_1,172.0,"{'team_1': [{'id': 66, 'self': '/api/players/6...",FIP,2026,Season 5,Center Court
1,7018,Triay/Brea - Fernandez/Gonzalez,women,1,Finals,0,2025-12-14,finished,"[{'team_1': '4', 'team_2': '6'}, {'team_1': '6...",team_2,129.0,"{'team_1': [{'id': 433, 'self': '/api/players/...",FIP,2026,Season 5,Center Court
2,6985,Tapia/Coello - Di Nenno/Stupaczuk,men,2,Semifinals,0,2025-12-13,finished,"[{'team_1': '7', 'team_2': '5'}, {'team_1': '6...",team_1,104.0,"{'team_1': [{'id': 66, 'self': '/api/players/6...",FIP,2026,Season 5,Center Court
3,6986,Navarro/Sanz - Chingotto/Galan,men,2,Semifinals,1,2025-12-13,finished,"[{'team_1': '2', 'team_2': '6'}, {'team_1': '4...",team_2,95.0,"{'team_1': [{'id': 77, 'self': '/api/players/7...",FIP,2026,Season 5,Center Court
4,7016,Triay/Brea - Ustero/Araujo,women,2,Semifinals,0,2025-12-13,finished,"[{'team_1': '6', 'team_2': '1'}, {'team_1': '6...",team_1,97.0,"{'team_1': [{'id': 433, 'self': '/api/players/...",FIP,2026,Season 5,Center Court


In [8]:
# Affiche les colonnes disponibles pour choisir les bonnes
print(df.dtypes)
print("\nValeurs manquantes:\n", df.isnull().sum())

id                      int64
name                   object
category               object
round                   int64
round_name             object
index                   int64
played_at              object
status                 object
score                  object
winner                 object
duration_minutes      float64
players                object
competition_source     object
season_year             int64
tournament_name        object
court                  object
dtype: object

Valeurs manquantes:
 id                      0
name                   82
category                0
round                   0
round_name              0
index                   0
played_at               0
status                  0
score                 375
winner                116
duration_minutes      624
players                 0
competition_source      0
season_year             0
tournament_name         0
court                 699
dtype: int64


In [9]:
# Créer les features à partir des colonnes disponibles
df2 = df.copy()

# Encoder la cible : team_1 gagne = 1, team_2 = 0
df2 = df2.dropna(subset=['winner'])
df2['target'] = (df2['winner'] == 'team_1').astype(int)

# Encoder les variables catégorielles
le_cat = LabelEncoder()
le_round = LabelEncoder()
le_court = LabelEncoder()
le_source = LabelEncoder()

df2['category_enc'] = le_cat.fit_transform(df2['category'])
df2['round_name_enc'] = le_round.fit_transform(df2['round_name'])
df2['court_enc'] = le_court.fit_transform(df2['court'].fillna('Unknown'))
df2['source_enc'] = le_source.fit_transform(df2['competition_source'])

# Features temporelles
df2['played_at'] = pd.to_datetime(df2['played_at'], errors='coerce')
df2['month'] = df2['played_at'].dt.month.fillna(0).astype(int)
df2['season_year_norm'] = df2['season_year'] - df2['season_year'].min()

# Features finales
FEATURES = ['category_enc', 'round', 'round_name_enc', 'index',
            'court_enc', 'source_enc', 'month', 'season_year_norm']

X = df2[FEATURES]
y = df2['target']

print("X shape:", X.shape)
print("Distribution cible:\n", y.value_counts())

X shape: (1144, 8)
Distribution cible:
 1    579
0    565
Name: target, dtype: int64


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# XGBoost
xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)

print("✅ Modèles entraînés !")

Train: (915, 8) | Test: (229, 8)
✅ Modèles entraînés !


In [11]:
for name, model in [("Random Forest", rf), ("XGBoost", xgb)]:
    preds = model.predict(X_test)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    print(f"\n=== {name} ===")
    print(classification_report(y_test, preds))
    print(f"ROC-AUC: {auc:.4f}")


=== Random Forest ===
              precision    recall  f1-score   support

           0       0.54      0.55      0.55       113
           1       0.56      0.55      0.55       116

    accuracy                           0.55       229
   macro avg       0.55      0.55      0.55       229
weighted avg       0.55      0.55      0.55       229

ROC-AUC: 0.6057

=== XGBoost ===
              precision    recall  f1-score   support

           0       0.72      0.77      0.75       113
           1       0.76      0.72      0.74       116

    accuracy                           0.74       229
   macro avg       0.74      0.74      0.74       229
weighted avg       0.74      0.74      0.74       229

ROC-AUC: 0.7935


In [12]:
os.makedirs('../models', exist_ok=True)
joblib.dump(rf, '../models/rf_classifier_matches.pkl')
joblib.dump(xgb, '../models/xgb_classifier_matches.pkl')
print("✅ Modèles sauvegardés dans /models !")

✅ Modèles sauvegardés dans /models !
